# Text Classification of Board-Game Q&A Posts

## Project overview

This notebook documents the approach used to solve a multi-class text classification task: predicting the `label` (board-game category) of a community Q&A post from its `text`, together with a handful of auxiliary numeric/date fields (`creation_date`, `score`, `views`, `answers`, `comments`, `favorites`).

It follows the same pipeline implemented in the final submission script (`solution1.py`):

1. Exploratory data analysis (class balance, text length).
2. Data cleaning: spam/garbage-post removal, numeric range filtering, and multivariate outlier detection.
3. Two text-cleaning variants feeding two different character n-gram TF-IDF representations, plus a small numeric feature derived from the post's creation year.
4. A couple of simple baseline models, followed by the final model — a weighted ensemble of three Logistic Regression classifiers.
5. Evaluation via macro-F1 and a confusion matrix on a held-out validation split.

> **Note:** running this notebook end to end requires a `train.json` file with the same schema as the one used for the final submission (`text`, `label`, `creation_date`, `score`, `views`, `answers`, `comments`, `favorites`) in the working directory.

In [ ]:
import json
import re

import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, confusion_matrix

import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42

# Global plotting style (looks more professional for the report)
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

## 1. Data loading & class distribution

We start by loading the labeled training data and dropping any rows without a label (they can't be used for supervised learning). The first thing worth checking in any classification problem is how balanced the classes are, since a strongly imbalanced dataset calls for extra care later on (class weighting, stratified splits, macro-averaged metrics, etc.).

In [ ]:
with open('train.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

df = pd.DataFrame(data)
df = df.dropna(subset=['label']).copy()
print('Rows loaded:', len(df))

plt.figure(figsize=(10, 6))
ax = sns.countplot(data=df, x='label', order=df['label'].value_counts().index, palette='viridis')

plt.title('Figure 1: Target class distribution (board games)', fontsize=14, fontweight='bold')
plt.ylabel('Number of posts', fontsize=12)
plt.xlabel('Class', fontsize=12)
plt.xticks(rotation=45)

# Annotate the exact count above each bar
for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='center', xytext=(0, 5), textcoords='offset points')

plt.tight_layout()
plt.savefig('figure1_class_distribution.png')  # saved to the working directory
plt.show()

## 2. Text length by class

Next we look at how long the posts are, broken down by class. This gives a quick sanity check on whether length alone carries some signal, and whether particular classes are systematically shorter or longer — which is also useful context when choosing n-gram ranges for the TF-IDF vectorizers later on.

In [ ]:
def clean_text_basic(text):
    """Very light cleaning used only for this exploratory word-count plot."""
    if not isinstance(text, str):
        return ""
    return re.sub(r'<[^>]+>', ' ', text)

df['clean_text'] = df['text'].apply(clean_text_basic)
df['word_count'] = df['clean_text'].apply(lambda x: len(x.split()))

plt.figure(figsize=(12, 6))
# showfliers=False hides extreme cases (1000+ word posts) so the plot stays readable
sns.boxplot(data=df, x='label', y='word_count', palette='Set2', showfliers=False)

plt.title('Figure 2: Text length distribution by class (extreme outliers hidden)', fontsize=14, fontweight='bold')
plt.ylabel('Word count per post', fontsize=12)
plt.xlabel('Game class', fontsize=12)
plt.xticks(rotation=45)

plt.tight_layout()
plt.savefig('figure2_text_length.png')
plt.show()

## 3. Data cleaning & feature engineering

Real-world community Q&A dumps contain a fair amount of noise: near-empty posts, garbled or non-alphabetic text, and repetitive spam. Before training any model we apply three cleaning steps, mirroring what the final submission script does:

1. **Range filtering** — engagement counters (`score`, `views`, `answers`, `comments`, `favorites`) should never be strongly negative; rows where any of them is below `-10` are almost certainly corrupted and are dropped.
2. **Spam / garbage detection (`is_spam`)** — a lightweight heuristic that flags a post as spam if, after stripping HTML tags: (a) it is shorter than 5 characters, (b) fewer than 30% of its non-space characters are alphabetic (garbled encodings, ASCII art, etc.), or (c) it is at least 8 words long but fewer than 15% of those words are unique (repetitive spam).
3. **Multivariate outlier detection** — the remaining engagement counters are log-transformed and fed into an `IsolationForest` (2% contamination) to catch multivariate anomalies that the simple range filter misses.

We also prepare two differently-cleaned versions of the text, since the two character-level TF-IDF representations used later benefit from different amounts of normalization:

- **`std_clean`** — aggressive normalization: strips HTML tags, URLs, punctuation, and standalone digits, then lowercases. This keeps only the "word skeleton" of the text, which works well for the shorter character n-gram ranges (2–4 and 4–6 characters).
- **`raw_clean`** — light cleaning: only strips HTML tags and lowercases. This preserves punctuation, numbers, and other surface artifacts that can still carry class-relevant signal, and is paired with a wider n-gram range (2–7 characters).

Finally, `extract_year_features` turns `creation_date` into two numeric features: the raw year (missing values are imputed with 2015, roughly the middle of the observed range) and a log-scaled "years since 2010" feature, capturing any drift in vocabulary or topic popularity over time.

In [ ]:
def is_spam(t):
    """Heuristic spam / garbage-post detector."""
    raw = re.sub(r'<[^>]+>', ' ', str(t)).strip()
    if len(raw) < 5:
        return True
    ns = [c for c in raw if not c.isspace()]
    if not ns or sum(1 for c in ns if c.isalpha()) / len(ns) < 0.30:
        return True
    words = raw.lower().split()
    if len(words) >= 8 and len(set(words)) / len(words) < 0.15:
        return True
    return False

def std_clean(t):
    """Aggressive cleaning: strips HTML, URLs, punctuation and standalone digits."""
    t = re.sub(r'<[^>]+>', ' ', str(t))
    t = re.sub(r'https?://\S+', ' ', t)
    t = re.sub(r'[^\w\s]', ' ', t, flags=re.UNICODE)
    t = re.sub(r'\b\d+\b', ' ', t)
    return ' '.join(t.lower().split())

def raw_clean(t):
    """Light cleaning: only strips HTML tags and lowercases."""
    return re.sub(r'<[^>]+>', ' ', str(t)).lower().strip()

def extract_year_features(frame):
    """Two numeric features derived from the post's creation date."""
    yr = pd.to_datetime(frame['creation_date'], errors='coerce').dt.year.fillna(2015).values
    return np.column_stack([yr, np.log1p(np.clip(yr - 2010, 0, 20))])


print('Rows before cleaning:', len(df))

# 1. Range filtering on the engagement counters
for col in ['score', 'views', 'answers', 'comments', 'favorites']:
    mask = pd.to_numeric(df[col], errors='coerce') < -10
    df = df[~mask.fillna(False)]
print('Rows after range filtering:', len(df))

# 2. Spam / garbage-post removal
df = df[~df['text'].apply(is_spam)].copy()
print('Rows after spam filtering:', len(df))

# 3. Multivariate outlier removal on log-transformed engagement counters
X_num_iso = pd.DataFrame({col: np.log1p(pd.to_numeric(df[col], errors='coerce').fillna(0).clip(lower=0))
                           for col in ['score', 'views', 'answers', 'comments', 'favorites']})
iso = IsolationForest(n_estimators=100, contamination=0.02, random_state=RANDOM_STATE)
df = df[iso.fit_predict(X_num_iso) == 1].copy().reset_index(drop=True)
print('Rows after IsolationForest filtering:', len(df))

## 4. Train / validation split

For local evaluation we hold out 15% of the cleaned data as a validation set, stratified by `label` so the class balance we saw above is preserved in both splits.

In [ ]:
train_df, val_df = train_test_split(
    df, test_size=0.15, stratify=df['label'], random_state=RANDOM_STATE
)

print('Training rows:', len(train_df))
print('Validation rows:', len(val_df))

## 5. Baseline models

Before building the final pipeline it's useful to establish a couple of simple baselines, both using a standard **word-level** TF-IDF representation of the lightly-cleaned text:

- **Multinomial Naive Bayes** — a fast, classic baseline for text classification.
- **Linear SVM (`LinearSVC`)** — typically a stronger baseline than Naive Bayes on high-dimensional sparse text features.

These give us a reference point for how much the more elaborate final pipeline actually buys us.

In [ ]:
vec_word = TfidfVectorizer(max_features=10000)
X_tr_w = vec_word.fit_transform(train_df['clean_text'])
X_va_w = vec_word.transform(val_df['clean_text'])

# --- Baseline 1: Multinomial Naive Bayes (word n-grams) ---
nb = MultinomialNB().fit(X_tr_w, train_df['label'])
f1_nb = f1_score(val_df['label'], nb.predict(X_va_w), average='macro')
print('Naive Bayes macro F1:', f1_nb)

# --- Baseline 2: Linear SVM (word n-grams) ---
svc = LinearSVC(class_weight='balanced', random_state=RANDOM_STATE).fit(X_tr_w, train_df['label'])
f1_svc = f1_score(val_df['label'], svc.predict(X_va_w), average='macro')
print('Linear SVM macro F1:', f1_svc)

## 6. Final model: weighted ensemble of character-level logistic regressions

The baselines above use a fairly generic word-level representation. The final model instead leans on **character n-grams**, which tend to be more robust to typos, inflected forms, and the informal writing style common in community Q&A posts, and combines three separate signals:

1. **`lr_std`** — Logistic Regression on a combination of two char n-gram TF-IDF matrices computed from the `std_clean` text: 2–4 grams and 4–6 grams (the 4–6 gram matrix is down-weighted by a factor of 0.7 before being combined, so it contributes a bit less than the shorter-range features).
2. **`lr_raw`** — Logistic Regression on a single, wider 2–7 char n-gram TF-IDF matrix computed from the lightly-cleaned `raw_clean` text.
3. **`lr_num`** — Logistic Regression on the two standardized year-based numeric features from `extract_year_features`.

Each model is trained independently and produces class probabilities; the final prediction is a **weighted average of the three probability vectors**, using weights (`ws`, `wr`, `wn`) that were tuned — together with the numeric model's regularization strength `C_num` — via a validation search (`best_params`). Blending probabilities rather than hard labels lets the stronger text models dominate while still letting the numeric/temporal signal nudge borderline cases.

In [ ]:
# Weights found via a validation search over the blend of the three sub-models,
# together with the regularization strength of the numeric-only model.
best_params = {'wn': 0.32, 'ws': 0.258, 'wr': 0.422, 'C_num': 5}

X_tr_std = train_df['text'].apply(std_clean).values
X_tr_raw = train_df['text'].apply(raw_clean).values
X_va_std = val_df['text'].apply(std_clean).values
X_va_raw = val_df['text'].apply(raw_clean).values

# Character n-gram TF-IDF vectorizers
c1s = TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 4), min_df=3, max_df=0.92,
                       sublinear_tf=True, max_features=25000)
c2s = TfidfVectorizer(analyzer='char_wb', ngram_range=(4, 6), min_df=3, max_df=0.92,
                       sublinear_tf=True, max_features=18000)
c1r = TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 7), min_df=2, max_df=0.92,
                       sublinear_tf=True, max_features=80000)

Xm_std_tr = sp.hstack([c1s.fit_transform(X_tr_std), c2s.fit_transform(X_tr_std) * 0.7], format='csr')
Xm_raw_tr = c1r.fit_transform(X_tr_raw)

num_yr_tr = extract_year_features(train_df)
sc = StandardScaler()
ny_tr_s = sc.fit_transform(num_yr_tr)

# Three independent Logistic Regression models
lr_std = LogisticRegression(C=10, max_iter=2000, class_weight='balanced', random_state=RANDOM_STATE)
lr_std.fit(Xm_std_tr, train_df['label'])

lr_raw = LogisticRegression(C=10, max_iter=2000, class_weight='balanced', random_state=RANDOM_STATE)
lr_raw.fit(Xm_raw_tr, train_df['label'])

lr_num = LogisticRegression(C=best_params['C_num'], max_iter=1000, class_weight='balanced', random_state=RANDOM_STATE)
lr_num.fit(ny_tr_s, train_df['label'])

classes = lr_std.classes_

# Transform the validation split with the vectorizers/scaler fitted on the training split
Xm_std_va = sp.hstack([c1s.transform(X_va_std), c2s.transform(X_va_std) * 0.7], format='csr')
Xm_raw_va = c1r.transform(X_va_raw)
ny_va_s = sc.transform(extract_year_features(val_df))

p_std = lr_std.predict_proba(Xm_std_va)
p_raw = lr_raw.predict_proba(Xm_raw_va)
p_num = lr_num.predict_proba(ny_va_s)

# Weighted blend of the three probability vectors
p_final = (p_std * best_params['ws']) + (p_raw * best_params['wr']) + (p_num * best_params['wn'])
y_pred_final = classes[np.argmax(p_final, axis=1)]

f1_final = f1_score(val_df['label'], y_pred_final, average='macro')
print('Final ensemble macro F1:', f1_final)

In [ ]:
models = ['Multinomial NB\n(word TF-IDF)', 'Linear SVM\n(word TF-IDF)', 'Final ensemble\n(char TF-IDF + numeric)']
scores = [f1_nb, f1_svc, f1_final]

plt.figure(figsize=(9, 6))
ax = sns.barplot(x=models, y=scores, palette='coolwarm')

plt.title('Figure 3: Macro F1 comparison across the models tried', fontsize=14, fontweight='bold')
plt.ylabel('Macro F1 score', fontsize=12)
plt.ylim(0.1, 1.0)  # zoom in on the y-axis so the differences are easier to see

for i, v in enumerate(scores):
    ax.text(i, v + 0.01, f"{v:.3f}", ha='center', fontweight='bold', fontsize=12)

plt.tight_layout()
plt.savefig('figure3_model_comparison.png')
plt.show()

## 7. Confusion matrix of the final model

Macro-F1 is a useful single number, but it hides *where* the model still gets confused. The confusion matrix below is built from the final ensemble's predictions on the validation split (the same `y_pred_final` computed in Section 6), so it reflects the actual model being submitted rather than a separate simplified model.

In [ ]:
cm = confusion_matrix(val_df['label'], y_pred_final, labels=classes)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=classes,
            yticklabels=classes)

plt.title('Figure 4: Confusion matrix (validation split, final ensemble)', fontsize=14, fontweight='bold')
plt.ylabel('True label', fontsize=12)
plt.xlabel('Predicted label', fontsize=12)
plt.xticks(rotation=45)

plt.tight_layout()
plt.savefig('figure4_confusion_matrix.png')
plt.show()

## 8. Conclusion

- The word-level baselines (Naive Bayes, Linear SVM) give a reasonable starting point, but are clearly outperformed by the character-level ensemble, which is more robust to noisy, informally-written text.
- Cleaning the data (spam filtering, range filtering, and multivariate outlier removal) matters: leaving obviously corrupted or spam-like rows in the training set would let the model waste capacity fitting noise instead of signal.
- Combining a "clean" character representation with a "raw" character representation and a small temporal signal, and blending their predicted probabilities, outperforms any single one of the three on its own.
- **Possible next steps:** a more exhaustive hyperparameter search over the TF-IDF vocabulary sizes and n-gram ranges, trying transformer-based sentence embeddings as a fourth signal in the ensemble, and inspecting the confusion matrix's largest off-diagonal cells to see whether particular classes would benefit from more training examples or additional features.